# 🐬 MySQL Features Showcase with Calliope AI

Explore MySQL-specific features and functions using Calliope AI with real sample databases.

## 🗄️ MySQL Databases Available

This notebook demonstrates features across **4 MySQL databases**:
- **sakila** - DVD rental store
- **employees** - HR database
- **world** - Geographic data
- **airportdb** - Aviation database

## 🎯 What We'll Cover

1. MySQL-specific data types
2. String functions
3. Date and time functions
4. JSON functions
5. Window functions (MySQL 8.0+)
6. Common Table Expressions (CTEs)
7. Full-text search
8. Performance optimization

## ⚙️ Configuration

**Run this cell first!** Sets up the Calliope API endpoint.

In [ ]:
# =============================================================================
# AUTO-INSTALL DEPENDENCIES
# Run this cell to scan the notebook and install required packages
# =============================================================================
%%calliope ask claude --format code
Scan this notebook for all Python imports and library dependencies.
Generate a shell script that:
1. Checks if each required package is installed
2. Installs any missing packages using pip
Output only the executable shell commands, no explanations.


In [ ]:
import os

# Configure Calliope Data Agent API endpoint
API_BASE_URL = 'http://localhost:5000'  # Local development
# API_BASE_URL = 'http://data-agent:5000'  # Docker/JupyterHub

os.environ['CALLIOPE_API_BASE_URL'] = API_BASE_URL
print(f"✅ Calliope API configured: {API_BASE_URL}")

## ✅ Verify MySQL Datasources

In [ ]:
import requests
import pandas as pd

response = requests.get('http://localhost:5000/api/datasources')
datasources = response.json()['datasources']

mysql_datasources = [ds for ds in datasources if ds['dialect'] == 'mysql']
print("✅ MySQL Datasources Available:")
for ds in mysql_datasources:
    print(f"  • {ds['id']}: {ds['name']}")

## 🔤 String Functions

MySQL has powerful string manipulation functions.

### CONCAT and String Manipulation

In [ ]:
%%calliope run-sql employees
SELECT 
    emp_no,
    CONCAT(first_name, ' ', last_name) as full_name,
    UPPER(CONCAT(LEFT(first_name, 1), LEFT(last_name, 1))) as initials,
    CONCAT('EMP-', LPAD(emp_no, 8, '0')) as employee_code
FROM employees
LIMIT 10

In [ ]:
%%calliope ask-sql world --sql-model anthropic
Find countries where the name contains 'United'

### Advanced String Functions

In [ ]:
%%calliope run-sql sakila
SELECT 
    title,
    LENGTH(title) as title_length,
    SUBSTRING(title, 1, 20) as short_title,
    REVERSE(title) as reversed_title,
    REPLACE(title, ' ', '_') as underscored
FROM film
WHERE LENGTH(title) > 20
LIMIT 10

## 📅 Date and Time Functions

MySQL excels at date/time manipulation.

### Date Calculations

In [ ]:
%%calliope run-sql employees
SELECT 
    emp_no,
    hire_date,
    YEAR(hire_date) as hire_year,
    QUARTER(hire_date) as hire_quarter,
    DAYNAME(hire_date) as hire_day_of_week,
    TIMESTAMPDIFF(YEAR, hire_date, CURDATE()) as years_employed,
    TIMESTAMPDIFF(MONTH, hire_date, CURDATE()) as months_employed,
    DATE_ADD(hire_date, INTERVAL 25 YEAR) as silver_anniversary
FROM employees
ORDER BY hire_date
LIMIT 15

In [ ]:
%%calliope ask-sql employees --sql-model anthropic
Show employees hired in December of any year

### Date Formatting

In [ ]:
%%calliope run-sql employees
SELECT 
    emp_no,
    hire_date,
    DATE_FORMAT(hire_date, '%W, %M %d, %Y') as formatted_date,
    DATE_FORMAT(hire_date, '%m/%d/%Y') as us_format,
    DATE_FORMAT(hire_date, '%d-%m-%Y') as uk_format
FROM employees
LIMIT 10

## 📊 Aggregate Functions and GROUP BY Extensions

In [ ]:
%%calliope run-sql world
SELECT 
    Continent,
    COUNT(*) as num_countries,
    SUM(Population) as total_population,
    AVG(Population) as avg_population,
    MIN(Population) as min_population,
    MAX(Population) as max_population,
    STD(Population) as std_dev_population,
    GROUP_CONCAT(DISTINCT GovernmentForm ORDER BY GovernmentForm SEPARATOR ', ') as government_types
FROM country
GROUP BY Continent
ORDER BY total_population DESC

## 🪟 Window Functions (MySQL 8.0+)

Powerful analytical functions for rankings, running totals, and more.

### Ranking Functions

In [ ]:
%%calliope run-sql employees
SELECT 
    e.emp_no,
    CONCAT(e.first_name, ' ', e.last_name) as name,
    d.dept_name,
    s.salary,
    RANK() OVER (PARTITION BY d.dept_name ORDER BY s.salary DESC) as dept_salary_rank,
    DENSE_RANK() OVER (PARTITION BY d.dept_name ORDER BY s.salary DESC) as dept_salary_dense_rank,
    ROW_NUMBER() OVER (PARTITION BY d.dept_name ORDER BY s.salary DESC) as row_num
FROM employees e
JOIN dept_emp de ON e.emp_no = de.emp_no
JOIN departments d ON de.dept_no = d.dept_no
JOIN salaries s ON e.emp_no = s.emp_no
WHERE de.to_date = '9999-01-01'
  AND s.to_date = '9999-01-01'
LIMIT 30

### Analytical Functions

In [ ]:
%%calliope run-sql world
SELECT 
    Name,
    Continent,
    Population,
    AVG(Population) OVER (PARTITION BY Continent) as continent_avg_pop,
    Population - AVG(Population) OVER (PARTITION BY Continent) as diff_from_avg,
    PERCENT_RANK() OVER (PARTITION BY Continent ORDER BY Population) as percentile_rank,
    NTILE(4) OVER (PARTITION BY Continent ORDER BY Population) as quartile
FROM country
WHERE Population > 0
LIMIT 30

### Running Totals and Moving Averages

In [ ]:
%%calliope run-sql sakila
SELECT 
    payment_date,
    amount,
    SUM(amount) OVER (ORDER BY payment_date) as running_total,
    AVG(amount) OVER (ORDER BY payment_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as moving_avg_7day,
    LAG(amount, 1) OVER (ORDER BY payment_date) as previous_payment,
    LEAD(amount, 1) OVER (ORDER BY payment_date) as next_payment
FROM payment
ORDER BY payment_date
LIMIT 20

## 🔗 Common Table Expressions (CTEs)

WITH clauses make complex queries more readable.

In [ ]:
%%calliope run-sql employees
WITH dept_stats AS (
    SELECT 
        d.dept_name,
        COUNT(*) as employee_count,
        AVG(s.salary) as avg_salary
    FROM departments d
    JOIN dept_emp de ON d.dept_no = de.dept_no
    JOIN salaries s ON de.emp_no = s.emp_no
    WHERE de.to_date = '9999-01-01'
      AND s.to_date = '9999-01-01'
    GROUP BY d.dept_no, d.dept_name
),
overall_stats AS (
    SELECT 
        AVG(avg_salary) as company_avg_salary
    FROM dept_stats
)
SELECT 
    ds.dept_name,
    ds.employee_count,
    ROUND(ds.avg_salary, 2) as dept_avg_salary,
    ROUND(os.company_avg_salary, 2) as company_avg_salary,
    ROUND(ds.avg_salary - os.company_avg_salary, 2) as diff_from_company_avg,
    ROUND((ds.avg_salary / os.company_avg_salary - 1) * 100, 2) as pct_diff
FROM dept_stats ds
CROSS JOIN overall_stats os
ORDER BY ds.avg_salary DESC

### Recursive CTEs

In [ ]:
%%calliope run-sql employees
-- Generate a sequence of years
WITH RECURSIVE years AS (
    SELECT 1985 as year
    UNION ALL
    SELECT year + 1
    FROM years
    WHERE year < 2000
)
SELECT 
    y.year,
    COUNT(e.emp_no) as employees_hired
FROM years y
LEFT JOIN employees e ON YEAR(e.hire_date) = y.year
GROUP BY y.year
ORDER BY y.year

## 🔀 CASE Statements and Conditional Logic

In [ ]:
%%calliope run-sql world
SELECT 
    Name,
    Population,
    CASE 
        WHEN Population > 100000000 THEN 'Very Large'
        WHEN Population > 50000000 THEN 'Large'
        WHEN Population > 10000000 THEN 'Medium'
        WHEN Population > 1000000 THEN 'Small'
        ELSE 'Very Small'
    END as size_category,
    LifeExpectancy,
    CASE
        WHEN LifeExpectancy IS NULL THEN 'Unknown'
        WHEN LifeExpectancy > 80 THEN 'Very High'
        WHEN LifeExpectancy > 70 THEN 'High'
        WHEN LifeExpectancy > 60 THEN 'Medium'
        ELSE 'Low'
    END as life_expectancy_category
FROM country
ORDER BY Population DESC
LIMIT 20

## 📦 Subqueries and Derived Tables

In [ ]:
%%calliope run-sql sakila
SELECT 
    c.name as category,
    COUNT(*) as film_count,
    AVG(f.rental_rate) as avg_rental_rate
FROM category c
JOIN film_category fc ON c.category_id = fc.category_id
JOIN film f ON fc.film_id = f.film_id
WHERE f.rental_rate > (
    SELECT AVG(rental_rate) 
    FROM film
)
GROUP BY c.category_id, c.name
ORDER BY film_count DESC

## 🔗 Advanced JOIN Techniques

In [ ]:
%%calliope run-sql employees
-- Self-join to find employees with same birth year
SELECT 
    e1.emp_no as emp1_no,
    CONCAT(e1.first_name, ' ', e1.last_name) as emp1_name,
    e2.emp_no as emp2_no,
    CONCAT(e2.first_name, ' ', e2.last_name) as emp2_name,
    YEAR(e1.birth_date) as birth_year
FROM employees e1
JOIN employees e2 ON YEAR(e1.birth_date) = YEAR(e2.birth_date)
    AND e1.emp_no < e2.emp_no
WHERE YEAR(e1.birth_date) = 1960
LIMIT 20

## 🎯 UNION Operations

In [ ]:
%%calliope run-sql sakila
-- Combine actor names and customer names
SELECT 'Actor' as type, first_name, last_name
FROM actor
WHERE first_name LIKE 'A%'
UNION ALL
SELECT 'Customer' as type, first_name, last_name
FROM customer
WHERE first_name LIKE 'A%'
ORDER BY first_name, last_name
LIMIT 30

## ⚡ Performance Tips with EXPLAIN

In [ ]:
%%calliope run-sql employees
EXPLAIN SELECT 
    e.emp_no,
    CONCAT(e.first_name, ' ', e.last_name) as name,
    d.dept_name,
    s.salary
FROM employees e
JOIN dept_emp de ON e.emp_no = de.emp_no
JOIN departments d ON de.dept_no = d.dept_no
JOIN salaries s ON e.emp_no = s.emp_no
WHERE de.to_date = '9999-01-01'
  AND s.to_date = '9999-01-01'
  AND s.salary > 80000
LIMIT 100

## 🤖 Using Calliope AI for MySQL-Specific Queries

In [ ]:
%%calliope ask-sql employees --sql-model anthropic
Using window functions, rank employees by salary within their department and show only those in the top 3

In [ ]:
%%calliope ask-sql world --sql-model anthropic
Create a recursive CTE that shows population growth if each country grew by 2% per year for 10 years

In [ ]:
%%calliope ask-sql sakila --sql-model anthropic --to-ai --model claude3
Analyze rental patterns using window functions and identify trends. What insights can you provide?

## 📊 Summary

This notebook demonstrated MySQL-specific features:

✅ **String Functions**: CONCAT, SUBSTRING, REPLACE, etc.  
✅ **Date/Time Functions**: DATE_FORMAT, TIMESTAMPDIFF, DATE_ADD  
✅ **Window Functions**: RANK, ROW_NUMBER, running totals  
✅ **CTEs**: WITH clauses, recursive queries  
✅ **Advanced Aggregates**: GROUP_CONCAT, STD  
✅ **Joins & Subqueries**: Self-joins, derived tables  
✅ **Performance**: EXPLAIN analysis  

## 🎯 Next Steps

- Compare with PostgreSQL features in `PostgreSQL-Features-Showcase.ipynb`
- Explore individual datasources: Sakila, Employees, World, AirportDB
- Build complex analytical queries with Calliope AI